# Implementing LoRA From Scratch — notebook thực hành

## Mở notebook trên Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USERNAME/dailyofdose/blob/main/notebooks/lora_from_scratch_vi.ipynb)

> Nếu repo của bạn đang ở GitHub, hãy thay `YOUR_GITHUB_USERNAME` bằng username/org thật và đổi `main` thành tên branch chứa notebook nếu cần. Link Colab có dạng: `https://colab.research.google.com/github/<owner>/<repo>/blob/<branch>/notebooks/lora_from_scratch_vi.ipynb`.

Notebook này được viết theo kiểu Google Colab để bạn có thể **chạy thử**, **sửa code**, và **hiểu cơ chế LoRA/QLoRA** dựa trên bài *Implementing LoRA From Scratch for Fine-tuning LLMs*.

## Mục tiêu

1. Hiểu vì sao fine-tuning toàn bộ mô hình lớn rất tốn tài nguyên.
2. Tự implement LoRA bằng PyTorch từ hai ma trận low-rank `A` và `B`.
3. Gắn LoRA vào một mạng neural nhỏ, freeze model gốc, và chỉ train tham số LoRA.
4. Merge trọng số LoRA vào layer gốc để inference không tăng latency.
5. Xem workflow PEFT của Hugging Face ở dạng optional, giống cách bạn sẽ dùng với LLM thật.

> Gợi ý Colab: Runtime → Change runtime type → chọn GPU nếu có. Phần PyTorch toy example chạy được cả trên CPU.

## 0. Cài thư viện

Nếu chạy trên Google Colab, phần LoRA from scratch chỉ cần `torch` có sẵn trong runtime. Chỉ cài thêm `transformers/peft` nếu bạn thật sự muốn thử phần PEFT optional ở cuối notebook.

> Không chạy `pip install -U torch` trong Colab trừ khi bạn biết rõ mình đang làm gì, vì việc upgrade riêng PyTorch có thể làm lệch CUDA version với `torchvision` và gây lỗi kiểu `PyTorch has CUDA Version=13.0 and torchvision has CUDA Version=12.8`.


In [ ]:
# Bỏ comment nếu bạn muốn thử phần Hugging Face PEFT optional.
# Không upgrade torch/torchvision ở đây để tránh mismatch CUDA trong Colab.
# %pip install -q -U transformers datasets peft accelerate evaluate "torchao>=0.16.0"
# Sau khi cài/upgrade package trong Colab, nên vào Runtime → Restart session rồi chạy lại notebook.


## 1. Imports, seed, device

Ta cố định seed để kết quả tương đối ổn định giữa các lần chạy.

In [ ]:
import math
import random
from dataclasses import dataclass

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

## 2. Ý tưởng LoRA trong thực tế PyTorch

Với một `nn.Linear(in_features, out_features)`, PyTorch lưu `weight` có shape:

```text
(out_features, in_features)
```

Forward mặc định là:

```text
y = x @ W.T + bias
```

LoRA thêm một nhánh cập nhật low-rank:

```text
y = x @ W.T + bias + scale * (x @ A @ B)
```

Trong đó:

- `A`: shape `(in_features, r)`
- `B`: shape `(r, out_features)`
- `r`: rank nhỏ, ví dụ `1`, `4`, `8`, `16`
- `scale = alpha / r`
- Ban đầu `B = 0`, nên nhánh LoRA trả về `0`; model vẫn giống model gốc trước khi fine-tune.

In [ ]:
class LoRAWeights(nn.Module):
    # Low-rank adapter cho một Linear layer.
    # Nhánh LoRA học delta_W ≈ A @ B thay vì học toàn bộ W.
    # Với input x có shape (..., in_features), output có shape (..., out_features).

    def __init__(self, in_features: int, out_features: int, rank: int = 4, alpha: float = 8.0):
        super().__init__()
        if rank <= 0:
            raise ValueError("rank must be positive")

        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.alpha = alpha
        self.scale = alpha / rank

        # A được init ngẫu nhiên nhỏ; B init bằng 0 để ban đầu LoRA không làm đổi output.
        self.A = nn.Parameter(torch.randn(in_features, rank) * 0.01)
        self.B = nn.Parameter(torch.zeros(rank, out_features))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return (x @ self.A @ self.B) * self.scale

    def merged_delta_weight(self) -> torch.Tensor:
        # Trả về delta weight cùng orientation với nn.Linear.weight: (out_features, in_features).
        return (self.A @ self.B).T * self.scale

### Kiểm tra nhanh shape và trạng thái ban đầu

Vì `B` được init bằng zero, output LoRA lúc đầu phải toàn zero.

In [ ]:
lora_debug = LoRAWeights(in_features=5, out_features=3, rank=2, alpha=4)
x_debug = torch.randn(7, 5)
y_debug = lora_debug(x_debug)

print("x shape:", x_debug.shape)
print("LoRA output shape:", y_debug.shape)
print("Max abs output before training:", y_debug.abs().max().item())
print("Merged delta weight shape:", lora_debug.merged_delta_weight().shape)

## 3. Tạo dataset nhỏ để minh họa fine-tuning

Ta dùng dữ liệu 2D rất nhỏ để notebook chạy nhanh:

- **Task gốc**: phân loại theo `x0 + x1 > 0`.
- **Task mới**: phân loại theo `x0 - x1 > 0`.

Quy trình mô phỏng:

1. Train model gốc trên task gốc.
2. Freeze toàn bộ model gốc.
3. Gắn LoRA vào vài layer.
4. Train chỉ LoRA trên task mới.

In [ ]:
def make_dataset(n: int, task: str):
    x = torch.randn(n, 2)
    if task == "sum":
        y = (x[:, 0] + x[:, 1] > 0).long()
    elif task == "diff":
        y = (x[:, 0] - x[:, 1] > 0).long()
    else:
        raise ValueError("task must be 'sum' or 'diff'")
    return x, y

x_base_train, y_base_train = make_dataset(2_000, "sum")
x_base_test, y_base_test = make_dataset(1_000, "sum")

x_new_train, y_new_train = make_dataset(2_000, "diff")
x_new_test, y_new_test = make_dataset(1_000, "diff")

base_train_loader = DataLoader(TensorDataset(x_base_train, y_base_train), batch_size=64, shuffle=True)
new_train_loader = DataLoader(TensorDataset(x_new_train, y_new_train), batch_size=64, shuffle=True)

## 4. Model gốc

Đây là một MLP nhỏ. Trong LLM thật, các `Linear layer` tương tự xuất hiện rất nhiều trong attention projection và feed-forward blocks.

In [ ]:
class TinyClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 32)
        self.fc2 = nn.Linear(32, 32)
        self.fc3 = nn.Linear(32, 2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)


def accuracy(model: nn.Module, x: torch.Tensor, y: torch.Tensor) -> float:
    model.eval()
    with torch.no_grad():
        logits = model(x.to(DEVICE))
        preds = logits.argmax(dim=-1).cpu()
    return (preds == y).float().mean().item()


def count_parameters(model: nn.Module):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

base_model = TinyClassifier().to(DEVICE)
print(base_model)
print("Total/trainable params:", count_parameters(base_model))

## 5. Pre-train model gốc trên task gốc

Đây là bước tương đương với việc ta đã có một pretrained model trước khi fine-tune.

In [ ]:
def train_classifier(model, loader, epochs=10, lr=1e-2):
    model.train()
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(1, epochs + 1):
        total_loss = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * xb.size(0)

        if epoch in {1, epochs}:
            print(f"epoch={epoch:02d} loss={total_loss / len(loader.dataset):.4f}")

train_classifier(base_model, base_train_loader, epochs=10, lr=1e-2)

print("Accuracy on original task:", accuracy(base_model, x_base_test, y_base_test))
print("Accuracy on new task before LoRA:", accuracy(base_model, x_new_test, y_new_test))

## 6. Gắn LoRA và freeze model gốc

Ta tạo wrapper `TinyClassifierWithLoRA`:

- Giữ nguyên các layer gốc `fc1`, `fc2`, `fc3`.
- Thêm nhánh LoRA song song cho `fc1` và `fc2`.
- Freeze toàn bộ tham số model gốc.
- Chỉ `A` và `B` trong LoRA được train.

Ta không gắn LoRA vào `fc3` để minh họa rằng không nhất thiết phải LoRA mọi layer.

In [ ]:
class TinyClassifierWithLoRA(nn.Module):
    def __init__(self, base: TinyClassifier, rank: int = 4, alpha: float = 8.0):
        super().__init__()
        self.base = base

        # Freeze pretrained/base model.
        for param in self.base.parameters():
            param.requires_grad = False

        self.lora_fc1 = LoRAWeights(base.fc1.in_features, base.fc1.out_features, rank=rank, alpha=alpha)
        self.lora_fc2 = LoRAWeights(base.fc2.in_features, base.fc2.out_features, rank=rank, alpha=alpha)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.base.fc1(x) + self.lora_fc1(x))
        x = self.relu(self.base.fc2(x) + self.lora_fc2(x))
        return self.base.fc3(x)

lora_model = TinyClassifierWithLoRA(base_model, rank=4, alpha=8).to(DEVICE)

print("Total/trainable params with LoRA:", count_parameters(lora_model))
for name, param in lora_model.named_parameters():
    if param.requires_grad:
        print("trainable:", name, tuple(param.shape))

## 7. Fine-tune chỉ LoRA trên task mới

Điểm quan trọng: optimizer chỉ nhận các parameter có `requires_grad=True`, tức là chỉ train LoRA.

In [ ]:
print("Before LoRA fine-tuning")
print("Accuracy on new task:", accuracy(lora_model, x_new_test, y_new_test))
print("Accuracy on original task:", accuracy(lora_model, x_base_test, y_base_test))

train_classifier(lora_model, new_train_loader, epochs=15, lr=5e-2)

print("\nAfter LoRA fine-tuning")
print("Accuracy on new task:", accuracy(lora_model, x_new_test, y_new_test))
print("Accuracy on original task:", accuracy(lora_model, x_base_test, y_base_test))

## 8. Merge LoRA vào weight gốc để inference

Một ưu điểm lớn của LoRA là khi deploy, ta có thể cộng `delta_weight` vào `Linear.weight` gốc:

```text
W_merged = W + delta_W
```

Sau khi merge, model thường không cần nhánh LoRA riêng nữa, nên inference có thể dùng architecture gốc.

In [ ]:
def merge_lora_into_base(lora_model: TinyClassifierWithLoRA) -> TinyClassifier:
    merged = TinyClassifier().to(DEVICE)
    merged.load_state_dict(lora_model.base.state_dict())

    with torch.no_grad():
        merged.fc1.weight += lora_model.lora_fc1.merged_delta_weight()
        merged.fc2.weight += lora_model.lora_fc2.merged_delta_weight()

    return merged

merged_model = merge_lora_into_base(lora_model)

print("LoRA wrapper accuracy on new task:", accuracy(lora_model, x_new_test, y_new_test))
print("Merged model accuracy on new task:", accuracy(merged_model, x_new_test, y_new_test))

# Kiểm tra output gần như giống nhau.
with torch.no_grad():
    sample = x_new_test[:10].to(DEVICE)
    max_diff = (lora_model(sample) - merged_model(sample)).abs().max().item()
print("Max logit difference between LoRA wrapper and merged model:", max_diff)

## 9. Tóm tắt số parameter tiết kiệm được

Với layer `fc1`:

- Full fine-tune cần train `out_features * in_features` parameter cho weight, cộng bias nếu có.
- LoRA chỉ train `in_features * r + r * out_features` parameter.

Với LLM, chênh lệch này rất lớn vì các matrix có kích thước hàng nghìn hoặc hàng chục nghìn chiều.

In [ ]:
def lora_param_saving(in_features, out_features, rank):
    full = in_features * out_features
    lora = in_features * rank + rank * out_features
    return full, lora, full / lora

for in_features, out_features, rank in [(2048, 12288, 16), (2048, 12288, 1), (32, 32, 4)]:
    full, lora, ratio = lora_param_saving(in_features, out_features, rank)
    print(
        f"W=({in_features}, {out_features}), r={rank}: "
        f"full={full:,}, lora={lora:,}, reduction≈{ratio:.1f}x"
    )

## 10. Optional: Workflow Hugging Face PEFT

Phần này giống workflow dùng trong thực tế với model từ Hugging Face:

1. Tạo `LoraConfig`.
2. Load hoặc tạo base model.
3. Gọi `get_peft_model(base_model, config)`.
4. Train bằng `Trainer` hoặc custom PyTorch loop.

### Vì sao cell PEFT không chạy mặc định?

PEFT phụ thuộc `transformers`, mà `transformers` có thể lazy-import thêm nhiều module phụ như `torchvision`. Trên một số Colab runtime, nếu `torch` và `torchvision` bị lệch CUDA major version, import có thể fail dù code của ta không dùng vision model. Vì vậy phần dưới đây được để dưới dạng **tham khảo/commented code** để notebook vẫn `Run all` được cho phần LoRA from scratch.

Nếu bạn gặp lỗi `PyTorch has CUDA Version=... and torchvision has CUDA Version=...`, cách sạch nhất là: Runtime → Disconnect and delete runtime → mở lại notebook → **không** upgrade `torch` riêng lẻ → chỉ cài thêm `transformers peft accelerate evaluate torchao>=0.16.0` nếu cần PEFT.


In [ ]:
# Optional compatibility check: cell này chỉ IN hướng dẫn, không tự cài package.
# Mục tiêu là tránh việc notebook tự thay đổi torch/torchvision trong Colab runtime.
import importlib.metadata as importlib_metadata
import re


def version_tuple(version: str):
    numbers = re.findall(r"\d+", version.split("+")[0])
    return tuple(int(number) for number in numbers[:3])


minimum_torchao = (0, 16, 0)
try:
    installed_torchao = importlib_metadata.version("torchao")
except importlib_metadata.PackageNotFoundError:
    installed_torchao = None

if installed_torchao is None:
    print("torchao is not installed. Nếu muốn chạy PEFT mới, cài: %pip install -q -U 'torchao>=0.16.0'")
elif version_tuple(installed_torchao) < minimum_torchao:
    print(f"torchao=={installed_torchao} may be too old for recent PEFT.")
    print("Trong Colab, chạy: %pip install -q -U 'torchao>=0.16.0' rồi Runtime → Restart session.")
else:
    print(f"torchao=={installed_torchao} looks compatible for recent PEFT.")


In [ ]:
# Optional PEFT reference only — KHÔNG chạy mặc định để tránh lỗi môi trường Colab.
# Khi môi trường đã sạch/tương thích, bạn có thể copy code bên dưới sang cell mới và chạy.

# from transformers import AutoModelForCausalLM, GPT2Config
# from peft import LoraConfig, TaskType, get_peft_model
#
# tiny_gpt2_config = GPT2Config(
#     vocab_size=100,
#     n_positions=64,
#     n_embd=32,
#     n_layer=2,
#     n_head=4,
# )
# hf_base_model = AutoModelForCausalLM.from_config(tiny_gpt2_config)
#
# peft_config = LoraConfig(
#     task_type=TaskType.CAUSAL_LM,
#     inference_mode=False,
#     r=4,
#     lora_alpha=8,
#     lora_dropout=0.05,
#     target_modules=["c_attn", "c_proj"],
# )
#
# peft_model = get_peft_model(hf_base_model, peft_config)
# peft_model.print_trainable_parameters()

print("PEFT reference cell is intentionally not executed. Copy the commented code to a new cell after installing compatible dependencies.")


## 11. QLoRA khác LoRA ở đâu?

LoRA giảm số parameter cần train bằng cách học hai ma trận low-rank `A` và `B`.

QLoRA đi thêm một bước: **quantize model gốc** xuống low-bit, thường là 4-bit, để giảm memory của phần frozen base model. Khi đó:

- Base model weight `W` được lưu/tính toán theo dạng quantized để tiết kiệm VRAM.
- LoRA adapter vẫn được train để học update theo task mới.
- Tổng memory thấp hơn LoRA thường, nên có thể fine-tune model lớn hơn trên GPU nhỏ hơn.

Trong thực tế, QLoRA với Hugging Face thường dùng `bitsandbytes` và `BitsAndBytesConfig`. Phần này phụ thuộc GPU/CUDA nên notebook không bật mặc định.

In [ ]:
# Skeleton QLoRA tham khảo; không chạy mặc định vì cần GPU/CUDA + bitsandbytes.
# !pip install -q bitsandbytes
# from transformers import BitsAndBytesConfig, AutoModelForCausalLM
#
# quant_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16,
# )
#
# qlora_base_model = AutoModelForCausalLM.from_pretrained(
#     "your-model-name",
#     quantization_config=quant_config,
#     device_map="auto",
# )
# qlora_model = get_peft_model(qlora_base_model, peft_config)

## 12. Checklist tự học

Bạn có thể thử thay đổi các giá trị sau rồi chạy lại notebook:

- `rank`: thử `1`, `2`, `4`, `8`, `16`.
- `alpha`: thử `rank`, `2 * rank`, `4 * rank`.
- Gắn LoRA thêm vào `fc3`.
- Đổi task mới từ `diff` sang một rule khác.
- So sánh output trước và sau khi merge LoRA.

Nếu bạn hiểu được vì sao `B = 0` làm output ban đầu của LoRA bằng 0, và vì sao merge cần transpose `(A @ B).T`, bạn đã nắm được phần cốt lõi của LoRA trong PyTorch.